# 03 · `[경기 단독]` 루키 시즌 타깃 생성 (절대 폭발력 + 시즌 코호트 상대 우위)

이 노트북은 **실전 NFL 경기 데이터(`games.csv`, `game_tracking_2023~25.csv`, `player_play.csv`)**만 다루며, 드래프트 실패 예방을 위한 2중 신입 평가 타깃의 **원본 값**을 만든다:
- **기준 ① 절대 폭발력**: 전체 NFL 신입 풀(포지션 내)에서 압도적인 경기 운동능력·주전 여부
- **기준 ② 시즌 코호트 상대 우위**: 같은 드래프트 시즌(`draft_year × position`) 신입 중 상대적으로 두드러지는 순위·Z-score

⚠️ **누수 방지 원칙**: 백분위·코호트 Z-score·`starter_flag`처럼 *다른 선수들의 값*으로 기준을 정하는 상대 라벨은 여기서 저장하지 않는다. 테스트셋 분할 이후 `RelativeTargetEncoder`를 **Train 선수로만 fit**해서 만든다 (마지막 셀에서 미리보기).

### [0] 실행 환경 부트스트랩 (로컬 / Colab 겸용)
- **데이터 범위**: `공통 환경`
- **작업 단계**: `환경 설정`
- **목적 및 원리**: `StopIteration` 없이 프로젝트 루트를 탐색하고 모듈을 불러온다.
- **확인할 내용**: `9/9 CSVs ready`.

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 환경 설정] 로컬/VS Code Colab 커널 겸용 루트 탐색
# =====================================================================
import importlib.util
import os
import sys
from pathlib import Path

# Google Drive 내 다른 경로에 저장소/데이터가 있다면 아래 주석을 풀고 경로를 지정하세요.
# os.environ["NFL_PROJECT_ROOT"] = "/content/drive/MyDrive/nfl-bdb2027"
# os.environ["NFL_DATA_DIR"] = "/content/drive/MyDrive/nfl-bdb2027-data"

def _is_colab_kernel() -> bool:
    if (
        "google.colab" in sys.modules
        or "COLAB_RELEASE_TAG" in os.environ
        or "COLAB_GPU" in os.environ
    ):
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except (ModuleNotFoundError, ValueError):
        return False

def find_project_root() -> Path:
    candidates: list[Path] = []
    if env_root := os.environ.get("NFL_PROJECT_ROOT"):
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents, cwd / "nfl-bdb2027"])

    in_colab = _is_colab_kernel()
    drive_root = Path("/content/drive")
    mydrive = drive_root / "MyDrive"

    if in_colab:
        if not mydrive.exists():
            try:
                from google.colab import drive
                drive.mount(str(drive_root), force_remount=False)
            except Exception as exc:
                print(f"[warn] Google Drive 자동 마운트 실패: {exc}")

        candidates.extend([
            Path("/content/nfl-bdb2027"),
            mydrive / "nfl-bdb2027",
            mydrive / "Colab Notebooks" / "nfl-bdb2027",
        ])

    seen: set[Path] = set()
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        if (p / "src" / "config.py").exists():
            return p.resolve()

    if in_colab:
        skip = {"drive", "sample_data", ".config", ".git", ".venv", "__pycache__"}
        depth1_dirs: list[Path] = []
        for base in (Path("/content"), mydrive, drive_root / "Shareddrives"):
            if not base.exists():
                continue
            try:
                children = [
                    d for d in base.iterdir()
                    if d.is_dir() and d.name not in skip and not d.name.startswith(".")
                ]
            except OSError:
                continue
            depth1_dirs.extend(children)
            for d in children:
                for cand in (d, d / "nfl-bdb2027"):
                    if (cand / "src" / "config.py").exists():
                        return cand.resolve()

        for d in depth1_dirs:
            try:
                for sub in d.iterdir():
                    if not sub.is_dir() or sub.name.startswith("."):
                        continue
                    low = sub.name.lower()
                    if "nfl" in low or "bdb" in low:
                        if (sub / "src" / "config.py").exists():
                            return sub.resolve()
                    elif low in {"ml", "work", "projects", "repos", "github", "colab", "colab notebooks", "study"}:
                        cand = sub / "nfl-bdb2027"
                        if (cand / "src" / "config.py").exists():
                            return cand.resolve()
            except OSError:
                continue

    diag = [f"src/config.py를 찾지 못했습니다 (현재 커널 cwd={cwd}, colab={in_colab})."]
    if in_colab:
        for base in (Path("/content"), mydrive):
            if base.exists():
                try:
                    entries = sorted(p.name for p in base.iterdir() if not p.name.startswith("."))[:20]
                except OSError:
                    entries = ["<접근 불가>"]
                diag.append(f"  - {base} 폴더 목록: {entries}")
        diag.extend([
            "",
            "[원인] VS Code에서 Colab 커널(ipykernel)을 선택하면 코드는 로컬 PC가 아니라",
            "       원격 Google Colab VM(/content)에서 실행되므로 로컬 폴더를 직접 읽을 수 없습니다.",
            "[해결 방법]",
            "  1) 로컬 PC 파일로 실행: VS Code 우측 상단 커널을 Colab 대신 로컬 '.venv'로 변경",
            "  2) Colab 커널로 실행: Google Drive(/content/drive/MyDrive/nfl-bdb2027)에 저장소를 두거나,",
            '     다른 Drive 경로에 있다면 이 셀 상단에 os.environ["NFL_PROJECT_ROOT"] = "/content/drive/MyDrive/..." 지정',
        ])
    else:
        diag.append("환경변수 NFL_PROJECT_ROOT를 지정하거나 프로젝트 폴더 안에서 실행하세요.")
    raise FileNotFoundError("\n".join(diag))

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.features import game_targets
config.setup()

### [1] 경기 메타데이터 로딩 (`games.csv` 시즌 조인 준비)
- **데이터 범위**: `경기 단독`
- **작업 단계**: `데이터 로딩`
- **목적 및 원리**: `game_id[:4]`는 1~2월 플레이오프에서 연도가 어긋나므로 항상 `games.csv`로 `season`과 `season_type`을 붙인다.
- **확인할 내용**: 1,002경기, 510명 정상 로딩.

In [ ]:
# =====================================================================
# [데이터: 경기 단독] [단계: 데이터 로딩] games / players 메타 로딩
# =====================================================================
games, players = game_targets.load_game_meta()
display(games.groupby(["season", "season_type"]).size().unstack()[["PRE", "REG", "POST"]])

### [2] 2GB 경기 트래킹 1회 청크 스트리밍 & 글리치 제거
- **데이터 범위**: `경기 단독`
- **작업 단계**: `데이터 정제` + `이상치 제거`
- **목적 및 원리**: 2,265만 행을 100만 행씩 스트리밍하면서 루키 시즌(`season == draft_year`)만 조기 필터링하고, 글리치(`s > 12.5`, `a > 12.0`, `|Δs/Δt| > 12.0`)를 제거한 뒤 세그먼트별 3축 운동학 피크(속도·추진·감속·구심가속·자세전환)를 뽑는다.
- **확인할 내용**: 루키 트래킹 선수 492명, 글리치 제거 건수.

In [ ]:
# =====================================================================
# [데이터: 경기 단독] [단계: 데이터 정제 & 이상치 제거] 1회 스트리밍 패스
# =====================================================================
peaks = game_targets.stream_rookie_play_peaks(games, players)
display(peaks.head(5))

### [3] 루키 경기 운동능력 원본 타깃
- **데이터 범위**: `경기 단독`
- **작업 단계**: `타깃 생성(피처/라벨 집계)`
- **목적 및 원리**:
  - 대표 속도 타깃은 플레이 수 n의 **상위 5% 플레이 평균**(`g_smax_top5pct_mean`, k = max(1, ⌈0.05·n⌉))이다. 고정 5개 평균은 많이 뛸수록 운 좋은 값이 쌓여 저절로 커지므로(출전량 편향), 평균 낼 개수를 n에 비례시켜 많이 뛸수록 튄 값이 묽어지게 한다.
  - 가속·감속·구심 가속은 p99 지표를 쓰고, 노출 신뢰도 가중치 `w = n / (n + 20)`을 부여한다.
  - 선수 **자기 기록만으로** 정해지는 원본 값만 저장한다. 포지션/코호트 백분위·Z-score는 분할 후 Train 기준으로 만든다.
- **확인할 내용**: 포지션별 최고속도 중앙값, 노출 기준(≥20) 충족 선수 수.

In [ ]:
# =====================================================================
# [데이터: 경기 단독] [단계: 타깃 생성] T3 운동능력 타깃 생성
# =====================================================================
ath = game_targets.build_rookie_athletic_targets(peaks, players)
display(ath[~ath["low_exposure"]].groupby("combine_position")[[
]].median().round(2))

### [4] 루키 역할 · 포지션별 NGS 생산성 원본 값 & 저장
- **데이터 범위**: `경기 단독`
- **작업 단계**: `타깃 생성` + `검증 및 저장`
- **목적 및 원리**: 정규시즌 스냅·경기 수와 포지션별 유효 NGS 생산성을 계산해 Parquet로 저장한다. `starter_flag`(상위 1/3)와 버스트/스틸 진단은 컷을 Train에서 정해야 하므로 저장하지 않는다.
- **확인할 내용**: 3개 Parquet 저장, 포지션별 스냅 분포.

In [ ]:
# =====================================================================
# [데이터: 경기 단독] [단계: 타깃 생성 & 저장] T4 생성 및 Parquet 저장
# =====================================================================
role = game_targets.build_rookie_role_targets(games, players)

peaks.to_parquet(config.PROCESSED_FILES["game_play_peaks"], index=False)
ath.to_parquet(config.PROCESSED_FILES["game_rookie_athletic"], index=False)
role.to_parquet(config.PROCESSED_FILES["game_rookie_role"], index=False)

display(role.groupby("combine_position")["rookie_snaps_reg"].describe()[["count", "50%", "75%", "max"]])

### [5] (미리보기) 상대 기준 라벨은 Train으로만 기준을 잡는다
- **데이터 범위**: `경기 단독` + 고정 테스트 분할(`data/splits/test_ids.csv`)
- **작업 단계**: `타깃 생성(상대 라벨)` + `누수 검증`
- **목적 및 원리**: `RelativeTargetEncoder`를 Train 선수로만 `fit`하고 Train·Test에 `transform`한다. 교란 테스트로 *테스트 선수 정답을 바꿔도 Train 라벨이 바뀌지 않음*을 확인한다.
- **확인할 내용**: 교란 테스트 통과, Train 기준 `starter_flag` 비율 ≈ 1/3.

In [ ]:
# =====================================================================
# [데이터: 경기 단독] [단계: 상대 라벨 미리보기 & 누수 검증]
# =====================================================================
test_ids = set(pd.read_csv(config.TEST_IDS_PATH)["nfl_id"])
tgt = ath.merge(role.drop(columns=["draft_year", "combine_position", "sub_position"]), on="nfl_id")
tr, te = tgt[~tgt["nfl_id"].isin(test_ids)], tgt[tgt["nfl_id"].isin(test_ids)]

enc = game_targets.RelativeTargetEncoder().fit(tr)          # Train 선수로만 기준값 계산
tr_lab, te_lab = enc.transform(tr), enc.transform(te)        # Train·Test 모두 같은 기준 적용
game_targets.check_relative_labels_ignore_holdout(tr, te)
print("[확인된 결과] 교란 테스트 통과: 테스트 정답을 바꿔도 Train 상대 라벨 불변")

display(pd.DataFrame({
    "train_starter_rate": tr_lab.groupby("combine_position")["starter_flag"].mean(),
    "test_starter_rate": te_lab.groupby("combine_position")["starter_flag"].mean(),
}).round(3))